# SAC vs PID Comparison on F-16

This notebook compares a trained SAC agent with a PID controller on the `LinearLongitudinalF16-v0` environment for angle-of-attack tracking.

## 1. Imports

In [ ]:
import gymnasium as gym
import numpy as np
import torch

from tensoraerospace.agent.pid import PID
from tensoraerospace.agent.sac import SAC
from tensoraerospace.benchmark.function import overshoot, settling_time, static_error
from tensoraerospace.signals.standart import unit_step
from tensoraerospace.utils import convert_tp_to_sec_tp, generate_time_period

## 2. Simulation Parameters

Configure the discretization, time horizon, and reference signal (5-degree unit step at t=10s).

In [ ]:
dt = 0.01  # Discretization
tp = generate_time_period(tn=20, dt=dt) # Time period
tps = convert_tp_to_sec_tp(tp, dt=dt)
number_time_steps = len(tp) # Number of time steps
reference_signals = np.reshape(unit_step(degree=5, tp=tp, time_step=10, output_rad=True), [1, -1]) # Reference signal

## 3. Environment Setup

Create the `LinearLongitudinalF16-v0` environment tracking the angle of attack (alpha).

In [ ]:
env = gym.make(
    "LinearLongitudinalF16-v0",
    number_time_steps=number_time_steps,
    use_reward=True,
    initial_state=[[0], [0], [0]],
    reference_signal=reference_signals,
    state_space=["theta", "alpha", "q"],
    output_space=["theta", "alpha", "q"],
    tracking_states=["alpha"],
)

env.reset()

(array([-0.9435269 , -0.3312959 ,  0.35280824], dtype=float32), {})

In [4]:
seed = 42
replay_size = 1000000
batch_size = 256
updates_per_step = 1
num_steps = 1000001

In [5]:
torch.manual_seed(seed)
np.random.seed(seed)

## 4. SAC Agent Setup and Training

Initialize the SAC agent and train for one episode.

In [ ]:
agent = SAC(env, memory_capacity=replay_size, hidden_size=32, device="cpu", verbose_histogram=False)

In [7]:
agent.train(num_episodes=1)

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [07:08<00:00,  2.33it/s]


## 5. Evaluation

Evaluate the trained SAC agent over multiple episodes and record rewards.

In [15]:
# Evaluation
state, info = env.reset()
done = False
total_rew = 0
while not done:
    action = agent.select_action(state)
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_rew+=reward
print(total_rew)

-118.81768990978958


In [23]:
num_eval_episodes = 3
episode_returns = []
episode_lengths = []

for episode_num in range(num_eval_episodes):
    state, info = env.reset()
    done = False
    total_rew = 0.0
    steps = 0

    while not done:
        action = agent.select_action(state)
        state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_rew += float(reward)
        steps += 1

    episode_returns.append(total_rew)
    episode_lengths.append(steps)

print(f"Episode total rewards: {episode_returns}")
print(f"Episode lengths: {episode_lengths}")


/Users/asmazaev/Projects/TensorAeroSpace/.venv/lib/python3.11/site-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Moviepy - Building video /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent/eval-episode-0.mp4.
Moviepy - Writing video /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent/eval-episode-0.mp4



Moviepy - Done !
Moviepy - video ready /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent/eval-episode-0.mp4
Moviepy - Building video /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent/eval-episode-1.mp4.
Moviepy - Writing video /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent/eval-episode-1.mp4



Moviepy - Done !
Moviepy - video ready /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent/eval-episode-1.mp4
Moviepy - Building video /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent/eval-episode-2.mp4.
Moviepy - Writing video /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent/eval-episode-2.mp4



Moviepy - Done !
Moviepy - video ready /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent/eval-episode-2.mp4
Moviepy - Building video /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent/eval-episode-3.mp4.
Moviepy - Writing video /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent/eval-episode-3.mp4



Moviepy - Done !
Moviepy - video ready /Users/asmazaev/Projects/TensorAeroSpace/example/pendulum-agent/eval-episode-3.mp4
Episode total rewards: deque([array([-118.63133], dtype=float32), array([-128.95366], dtype=float32), array([-117.85647], dtype=float32), array([-124.018814], dtype=float32)], maxlen=100)
Episode lengths: deque([array([200], dtype=int32), array([200], dtype=int32), array([200], dtype=int32), array([200], dtype=int32)], maxlen=100)


## 6. Results

Plot the transient response of alpha vs the reference signal.

In [ ]:
env.unwrapped.model.plot_transient_process('alpha', tps, reference_signals[0], to_deg=True, figsize=(15,4))